<a href="https://colab.research.google.com/github/surajprakash-data-science/MMM-Using-Google-Meridian/blob/main/MMM_using_Meridian.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade google-meridian[colab,and-cuda,schema]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 74.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.6/503.6 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 952.4/952.4 kB 61.2 MB/s eta 0:00:00
  Attempting uninstall: natsort
    Found existing installation: natsort 8.4.0
    Uninstalling natsort-8.4.0:
      Successfully uninstalled natsort-8.4.0
  Attempting uninstall: arviz
    Found existing installation: arviz 0.22.0
    Uninstalling arviz-0.22.0:
      Successfully uninstalled arviz-0.22.0


In [ ]:
import IPython
from meridian import constants
from meridian.analysis import analyzer
from meridian.analysis import optimizer
from meridian.analysis import summarizer
from meridian.analysis import visualizer
from meridian.analysis.review import reviewer
from meridian.data import data_frame_input_data_builder
from meridian.model import model
from meridian.model import prior_distribution
from meridian.model import spec
from meridian.model.eda import meridian_eda
from meridian.schema.serde import meridian_serde
import numpy as np
import pandas as pd
# check if GPU is available
from psutil import virtual_memory
import tensorflow as tf
import tensorflow_probability as tfp

ram_gb = virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))
print(
    'Num GPUs Available: ',
    len(tf.config.experimental.list_physical_devices('GPU')),
)
print(
    'Num CPUs Available: ',
    len(tf.config.experimental.list_physical_devices('CPU')),
)

Your runtime has 13.6 gigabytes of available RAM

Num GPUs Available:  1
Num CPUs Available:  1


In [ ]:
import pandas as pd
import numpy as np

# Set seed for reproducibility
np.random.seed(42)

# Create a date range for 2 years (weekly)
dates = pd.date_range(start='2024-01-01', periods=104, freq='W')

# Generate Media Spend (with some randomness)
google_search_spend = np.random.normal(1500, 300, 104).clip(500)
meta_social_spend = np.random.normal(2000, 500, 104).clip(500)
tiktok_video_spend = np.random.normal(1000, 400, 104).clip(200)

# Generate Impressions (proportional to spend with some noise)
google_search_impressions = (google_search_spend * np.random.uniform(5, 8, 104) + np.random.normal(0, 5000, 104)).astype(int).clip(10000)
meta_social_impressions = (meta_social_spend * np.random.uniform(7, 10, 104) + np.random.normal(0, 7000, 104)).astype(int).clip(15000)
tiktok_video_impressions = (tiktok_video_spend * np.random.uniform(6, 9, 104) + np.random.normal(0, 4000, 104)).astype(int).clip(8000)

# Generate Control Variables
promo_active = np.random.choice([0, 1], size=104, p=[0.8, 0.2])
avg_price = np.where(promo_active == 1, 35, 50)
organic_traffic = np.random.normal(3000, 500, 104)

# Create the Revenue (Target) based on a formula + noise
# Formula: Base + (Google * 1.5) + (Meta_Lagged * 1.2) + (TikTok * 0.8) + (Promo_Effect)
base_revenue = 5000
revenue = (
    base_revenue +
    (google_search_spend * 2.2) +
    (pd.Series(meta_social_spend).shift(1).fillna(1000) * 1.8) + # Meta has a 1-week lag
    (tiktok_video_spend * 1.1) +
    (promo_active * 5000) +
    (np.random.normal(0, 1000, 104)) # Adding noise
)

# Create DataFrame
df = pd.DataFrame({
    'time': dates,
    'revenue': revenue.astype(int),
    'google_search_spend': google_search_spend.astype(int),
    'google_search_impressions': google_search_impressions,
    'meta_social_spend': meta_social_spend.astype(int),
    'meta_social_impressions': meta_social_impressions,
    'tiktok_video_spend': tiktok_video_spend.astype(int),
    'tiktok_video_impressions': tiktok_video_impressions,
    'organic_traffic': organic_traffic.astype(int),
    'promo_active': promo_active,
    'avg_price': avg_price
})

# Save to CSV
df.to_csv('marketing_mix_data.csv', index=False)
print("Dummy data for MMM project generated successfully!")

Dummy data for MMM project generated successfully!


In [ ]:
df.head()

,time,revenue,google_search_spend,google_search_impressions,meta_social_spend,meta_social_impressions,tiktok_video_spend,tiktok_video_impressions,organic_traffic,promo_active,avg_price
0,2024-01-07,12298,1649,16711,1919,16305,1206,12127,2958,0,50
1,2024-01-14,14088,1458,10000,2202,16055,2541,20952,3558,0,50
2,2024-01-21,14814,1694,10000,2943,30212,1228,10223,3171,0,50
3,2024-01-28,13353,1956,17542,2087,35173,1454,8000,3228,0,50
4,2024-02-04,19571,1429,10000,2128,18931,1381,8374,3284,1,35


In [ ]:
df.dtypes

,0
time,datetime64[ns]
revenue,int64
google_search_spend,int64
google_search_impressions,int64
meta_social_spend,int64
meta_social_impressions,int64
tiktok_video_spend,int64
tiktok_video_impressions,int64
organic_traffic,int64
promo_active,int64


## Data Notes

This synthetic dataset represents weekly marketing and sales data for a hypothetical product over two years (104 weeks). It includes:

*   **`date`**: Weekly date.
*   **`revenue`**: The target variable, representing weekly revenue.
*   **`google_search_spend`**, **`meta_social_spend`**, **`tiktok_video_spend`**: Weekly advertising spend on different platforms.
*   **`google_search_impressions`**, **`meta_social_impressions`**, **`tiktok_video_impressions`**: Weekly impressions generated by advertising on different platforms.
*   **`organic_traffic`**: Baseline organic website traffic, acting as a control variable.
*   **`promo_active`**: A binary flag (0 or 1) indicating if a promotion was active during that week.
*   **`avg_price`**: The average price of the product, which varies based on promotion activity.

The `revenue` is generated based on a formula that incorporates media spend (with a 1-week lag for Meta social spend), promotional effects, and random noise to simulate real-world variability.

## Define Column Categories

To prepare the data for Marketing Mix Modeling, we'll categorize the DataFrame columns into several groups:

*   **KPI (Key Performance Indicator)**: The primary metric we want to model (`revenue`).
*   **Media**: Variables related to advertising spend and impressions across different platforms.
*   **Non-Media**: Control variables that are not directly media-related but can influence the KPI (e.g., `promo_active`, `avg_price`).
*   **Organic**: Variables representing organic growth or baseline activity (e.g., `organic_traffic`).

# **Defining Data**

In [ ]:
# Define the column categories
kpi_col = ['revenue']

media_spend_cols = [
    'google_search_spend',
    'meta_social_spend',
    'tiktok_video_spend'
]

media_impressions_cols = [
    'google_search_impressions',
    'meta_social_impressions',
    'tiktok_video_impressions'
]

non_media_control_cols = [
    'promo_active'
]

organic_cols = [
    'organic_traffic'
]

print("Column categories defined successfully.")

Column categories defined successfully.


In [ ]:
builder = data_frame_input_data_builder.DataFrameInputDataBuilder(
    kpi_type='revenue',
    default_time_column='time',
    default_kpi_column='revenue'
    #,default_geo_column='geo'
)

builder.with_kpi(df, kpi_col='revenue')
builder.with_controls(df, control_cols=non_media_control_cols)
builder.with_media(
    df=df,
    media_cols=[
        'google_search_impressions',
        'meta_social_impressions',
        'tiktok_video_impressions'
    ],
    media_spend_cols=[
        'google_search_spend',
        'meta_social_spend',
        'tiktok_video_spend'
    ],
    media_channels=[
        'Google Search',
        'Meta Social',
        'TikTok Video'
    ]
)

# 4. Add organic channels using parallel lists
builder.with_organic_media(
    df=df,
    organic_media_cols=['organic_traffic'],
    organic_media_channels=['Organic Traffic']
)

input_data = builder.build()

/usr/local/lib/python3.12/dist-packages/meridian/data/input_data.py:517: UserWarning: Revenue from the `kpi` data is used when `kpi_type`=`revenue`. `revenue_per_kpi` is ignored.
  warnings.warn(


## Data Definition Notes

The `InputData` object has been successfully created using the `DataFrameInputDataBuilder`. This object encapsulates the marketing mix data, including:

*   **KPI**: `revenue`
*   **Time Column**: `time`
*   **Media Spend Columns**: `google_search_spend`, `meta_social_spend`, `tiktok_video_spend`
*   **Media Impressions Columns**: `google_search_impressions`, `meta_social_impressions`, `tiktok_video_impressions`
*   **Control Columns**: `promo_active`, `avg_price`
*   **Organic Columns**: `organic_traffic`

This `InputData` object is now ready for further analysis and model building within the Meridian framework.

# **Model configuration**

In [ ]:
roi_mu = 0.2  # Mu for ROI prior for each media channel.
roi_sigma = 0.9  # Sigma for ROI prior for each media channel.

prior = prior_distribution.PriorDistribution(
    roi_m=tfp.distributions.LogNormal(roi_mu, roi_sigma, name=constants.ROI_M)
)
model_spec = spec.ModelSpec(prior=prior, enable_aks=True)

mmm = model.Meridian(input_data=input_data, model_spec=model_spec)

/usr/local/lib/python3.12/dist-packages/meridian/model/model.py:103: UserWarning: In a nationally aggregated model, the `media_effects_dist` will be reset to `normal`.
  warnings.warn(


In [ ]:
mmm.sample_prior(500)

/usr/local/lib/python3.12/dist-packages/meridian/model/prior_distribution.py:1329: UserWarning: Hierarchical distribution parameters must be deterministically zero for national models. tau_g_excl_baseline has been automatically set to Deterministic(0).
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/meridian/model/prior_distribution.py:1329: UserWarning: Hierarchical distribution parameters must be deterministically zero for national models. eta_m has been automatically set to Deterministic(0).
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/meridian/model/prior_distribution.py:1329: UserWarning: Hierarchical distribution parameters must be deterministically zero for national models. eta_rf has been automatically set to Deterministic(0).
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/meridian/model/prior_distribution.py:1329: UserWarning: Hierarchical distribution parameters must be deterministically zero for national models. eta_om has been automatically se

## Model Configuration Notes

-   **ROI Prior**: A prior distribution for Return on Investment (ROI) has been set for each media channel. Specifically, a `LogNormal` distribution with `roi_mu = 0.2` and `roi_sigma = 0.9` is used.
-   **Model Specification**: A `ModelSpec` object was created, incorporating the defined ROI prior and enabling Automatic Kalman Smoothing (`enable_aks=True`).
-   **Meridian Model Initialization**: The `Meridian` model was initialized using the `input_data` object (created in the data definition step) and the `model_spec` object. This sets up the framework for the Marketing Mix Model.

# **EDA**

In [ ]:
eda = meridian_eda.MeridianEDA(mmm)
filename = 'eda_report.pdf'
meridian_root = '/content/'
eda.generate_and_save_report(filename=filename, filepath=meridian_root)
IPython.display.HTML(filename=f'{meridian_root}/{filename}')

1. The Critical Issue: Relationship Among Variables (The "FAIL")
This is the most important part of the report. The model has flagged a Multicollinearity error.
The Problem: The variables promo_active and avg_price have a pairwise correlation of -0.999 (nearly perfect negative correlation).
The VIF (Variance Inflation Factor): Both variables show a VIF of "inf" (Infinite).
What this means: In your data, every time a promotion is active, the price drops by a fixed, predictable amount. Because they move perfectly together, the model cannot distinguish which one is actually driving sales.
Recommended Action: You cannot include both variables in the model. You must either:
Drop one of them.
Combine them into a single "Price with Promo" index.
If you keep both, the model will likely fail to converge or give you nonsensical results.
2. Individual Explanatory Variables (Warnings)
This section looks at the "signal" in your data.
Low Variability (The "Zero Std Dev" Warning): After removing outliers, promo_active and avg_price show 0.000 variability.
This suggests that for the vast majority of your 104 weeks, these values never changed. If a variable doesn't change, the model can't learn how it affects the KPI.
Outliers: The report flagged several "Extreme" weeks:
Google Search had a massive spike on Sept 28, 2025.
TikTok Video had a spike on Jan 14, 2024.
Action: You should check these dates. Was there a data entry error? Or was there a viral moment or a massive tracking glitch? If these spikes are "fake," they will skew your ROI results.
3. Spend and Media Units
This section evaluates how your budget is distributed and if you have enough data to support the model's complexity.
Spend Share:
Meta Social: 44.9%
Google Search: 32.3%
TikTok Video: 22.8%
Insight: Your spend is relatively well-distributed. None of the channels are so small (e.g., <1%) that they would be impossible to measure.
Data Adequacy:
Your Ratio is 9.45. Meridian generally looks for a ratio where data points significantly outnumber the parameters being estimated.
With 104 weeks (2 years) of data and only 1 geo (national), you are in a "safe" zone, but you don't have a massive amount of "extra" data.
Cost Per Unit Outlier:
On Feb 16, 2025, the TikTok Cost Per Media Unit spiked to 0.206.
Action: Verify if TikTok actually became twice as expensive that week, or if the "Media Units" (Impressions/Clicks) were under-reported.
4. Summary of Data Stats
n_geos: 1 (This is a National-level model).
n_times: 104 (You have exactly 2 years of weekly data).
n_parameters: 11 (The model is trying to calculate 11 different "moving parts").
Knot count: 5 (This is used to control for seasonality/trends).

# **Fit Model**

In [ ]:
mmm.sample_posterior(
    n_chains=5, n_adapt=1000, n_burnin=500, n_keep=1000, seed=0
)

/usr/local/lib/python3.12/dist-packages/arviz/data/inference_data.py:157: UserWarning: trace group is not defined in the InferenceData scheme
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/arviz/data/inference_data.py:1647: UserWarning: trace group is not defined in the InferenceData scheme
  warnings.warn(


In [ ]:
health_summary = reviewer.ModelReviewer(mmm).run()

filename = 'health_card.html'
health_summary.output_model_health_card(filename=filename, filepath=meridian_root)
IPython.display.HTML(filename=f'{meridian_root}{filename}')

/tmp/ipykernel_440/879150696.py:1: DeprecationWarning: The `meridian` argument is deprecated. Please use `model_context` and `inference_data` instead.
  health_summary = reviewer.ModelReviewer(mmm).run()
/usr/local/lib/python3.12/dist-packages/meridian/analysis/analyzer.py:102: UserWarning: The `aggregate_geos` argument is ignored in the national model. It will be reset to `True`.
  warnings.warn(


Metric check,Status,Recommended action
Convergence,Pass,"The model has likely converged, as all parameters have R-hat values < 1.2."
Baseline,Pass,The posterior probability that the baseline is negative is 0.00. We recommend visually inspecting the baseline time series in the Model Fit charts to confirm this.
Bayesian p-value,Pass,The Bayesian posterior predictive p-value is 1.00. The observed total outcome is consistent with the model's posterior predictive distribution.
Goodness of fit,Pass,"R-squared = 0.7140, MAPE = 0.0872, and wMAPE = 0.0815. These goodness-of-fit metrics are intended for guidance and relative comparison."
Prior-posterior shift,Pass 3/3 channels passed,The model has successfully learned from the data. This is a positive sign that your data was informative.


# **Run model diagnostics**

In [ ]:
model_diagnostics = visualizer.ModelDiagnostics(mmm)
model_diagnostics.plot_rhat_boxplot()

alt.LayerChart(...)

In [ ]:
model_fit = visualizer.ModelFit(mmm)
model_fit.plot_model_fit()

/usr/local/lib/python3.12/dist-packages/meridian/analysis/analyzer.py:102: UserWarning: The `aggregate_geos` argument is ignored in the national model. It will be reset to `True`.
  warnings.warn(


alt.LayerChart(...)

# **model results & two-page output**

In [ ]:
mmm_summarizer = summarizer.Summarizer(mmm)

In [ ]:
filepath = meridian_root
start_date = df['time'].min().strftime('%Y-%m-%d') # Use the earliest date from the dataframe
end_date = df['time'].max().strftime('%Y-%m-%d') # Use the latest date from the dataframe
mmm_summarizer.output_model_results_summary(
    'summary_output.html', filepath, start_date, end_date
)

/usr/local/lib/python3.12/dist-packages/meridian/analysis/analyzer.py:102: UserWarning: The `aggregate_geos` argument is ignored in the national model. It will be reset to `True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/meridian/analysis/analyzer.py:102: UserWarning: The `aggregate_geos` argument is ignored in the national model. It will be reset to `True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/meridian/analysis/analyzer.py:2700: UserWarning: Effectiveness is not reported because it does not have a clear interpretation by time period.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/meridian/analysis/analyzer.py:352: UserWarning: Setting `use_kpi=True` has no effect when `kpi_type=REVENUE` since in this case, KPI is equal to revenue.
  warnings.warn(


In [ ]:
IPython.display.HTML(filename=f'{meridian_root}/summary_output.html')

Dataset,R-squared,MAPE,wMAPE
All Data,0.71,9%,8%


# **Report**

In [ ]:
%%time
budget_optimizer = optimizer.BudgetOptimizer(mmm)
optimization_results = budget_optimizer.optimize()

CPU times: user 1min 7s, sys: 3.11 s, total: 1min 11s
Wall time: 1min 7s


In [ ]:
filepath = meridian_root
optimization_results.output_optimization_summary(
    'optimization_output.html', filepath
)

In [ ]:
IPython.display.HTML(filename=f'{meridian_root}/optimization_output.html')

Channel,Non-optimized spend,Optimized spend
Meta Social,45%,45%
TikTok Video,23%,30%
Google Search,32%,26%


In [ ]:
# Grant Colab access to create Google Sheet in your Drive
from google.colab import auth

_SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive",
]
credentials = auth.authenticate_user(_SCOPES)

In [ ]:
!pip install google-meridian[scenarioplanner,and-cuda]

In [ ]:
# Now, run this cell and we can test if all dependencies are appropratly installed
import tensorflow_probability as tfp
import pandas as pd

from meridian import constants
from meridian.data import data_frame_input_data_builder
from meridian.model import model
from meridian.model import prior_distribution
from meridian.model import spec

from meridian.schema.processors import model_fit_processor
from meridian.schema.processors import marketing_processor
from meridian.schema.processors import budget_optimization_processor
from meridian.schema.utils import date_range_bucketing
from meridian.schema.serde import meridian_serde

from scenarioplanner.converters import sheets
from scenarioplanner.converters.dataframe import dataframe_model_converter
from scenarioplanner import mmm_ui_proto_generator as mmm_ui_gen
from scenarioplanner.linkingapi import url_generator

In [ ]:
# --- FAST SETUP ---
spreadsheet_name = "Meridian Fast Report"
opt_name = "Fast Demo"
include_non_paid = True

# Speed Tip: Setting this to False avoids heavy Reach/Frequency curve calculations
use_opt_freq = False

# Speed Tip: Only use one generator (e.g., Quarterly) to save processing time
time_generators = [date_range_bucketing.QuarterlyDateRangeGenerator]

# 1. Define Constraints (Condensed)
constraints = [
    budget_optimization_processor.ChannelConstraintRel(c, 1.0, 1.0)
    for c in mmm.input_data.get_all_paid_channels()
]

# 2. Build Specs
budget_spec = budget_optimization_processor.BudgetOptimizationSpec(
    optimization_name=opt_name,
    grid_name="fast-grid",
    constraints=constraints,
    use_optimal_frequency=use_opt_freq,
)

summary_spec = marketing_processor.MediaSummarySpec(include_non_paid_channels=include_non_paid)

# 3. Generate Proto & Convert (Combined)
print("Processing model data (Fast Mode)...")
mmm_proto = mmm_ui_gen.create_mmm_ui_data_proto(
    mmm=mmm,
    specs=[
        model_fit_processor.ModelFitSpec(),
        marketing_processor.MarketingAnalysisSpec(media_summary_spec=summary_spec),
        budget_spec,
    ],
    time_breakdown_generators=time_generators,
)

dataframes = dataframe_model_converter.DataFrameModelConverter(mmm_proto)()

# 4. Upload & Link
spreadsheet = sheets.upload_to_gsheet(dataframes, credentials, spreadsheet_name=spreadsheet_name)
url = url_generator.create_report_url(spreadsheet)

print(f"Done! Spreadsheet: {spreadsheet.url}")
from IPython.display import HTML
HTML(f'<a href="{url}" style="font-size:20px;">Click here to open Looker Studio Report</a>')

Processing model data (Fast Mode)...


/usr/local/lib/python3.12/dist-packages/meridian/analysis/analyzer.py:2115: UserWarning: `split_by_holdout_id` is True but `holdout_id` is `None`. Data will not be split.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/meridian/analysis/analyzer.py:102: UserWarning: The `aggregate_geos` argument is ignored in the national model. It will be reset to `True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/meridian/analysis/analyzer.py:102: UserWarning: The `aggregate_geos` argument is ignored in the national model. It will be reset to `True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/meridian/analysis/analyzer.py:352: UserWarning: Setting `use_kpi=True` has no effect when `kpi_type=REVENUE` since in this case, KPI is equal to revenue.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/meridian/analysis/analyzer.py:352: UserWarning: Setting `use_kpi=True` has no effect when `kpi_type=REVENUE` since in this case, KPI is equal to revenue.
  warnings.wa

Done! Spreadsheet: https://docs.google.com/spreadsheets/d/11pklPriidMpUaTeOQQzvdl41ZEFTdSYdZ6Ce0U6BWQY/edit
